# Real-time EV Charging Planner — Denmark

This notebook tells you **when to charge your EV today (and tomorrow)** based on live CO₂ data from the Danish grid.

**What it does:**
1. Fetches the **latest real-time CO₂ intensity** from Energi Data Service (5-min resolution)
2. Fetches the **24-hour CO₂ forecast** (`CO2EmisProg`) so you can plan ahead
3. Calculates your **optimal charging window** given your car's battery size and charging speed
4. Scores every hour on a **0–100 green scale** and shows you a colour-coded charging schedule

**Data source:** [Energi Data Service](https://www.energidataservice.dk/) — the official Danish TSO open API.  
**Metric:** `CO2Emission` (g CO₂eq / kWh) — lower = greener = better time to charge.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from datetime import datetime, timedelta, timezone
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded.')

## Configuration
Set your **location**, **car battery**, and **charging preferences** below.

In [ ]:
# ============================================================
#  USER SETTINGS — change these to match your setup
# ============================================================

# --- Location ---
YOUR_CITY = "Aarhus"        # Your Danish city

# --- EV / Charger ---
BATTERY_CAPACITY_KWH = 77   # Your EV's usable battery capacity (kWh)
                             # Examples: Tesla Model 3 LR=75, VW ID.4=77, Nissan Leaf=40
CHARGER_POWER_KW     = 11   # Your home charger output (kW)
                             # Typical: 7 kW (32A single-phase), 11 kW (3-phase), 22 kW
CURRENT_SOC_PCT      = 30   # Current state of charge (%) — how full is the battery now?
TARGET_SOC_PCT       = 80   # Target state of charge (%) — don't charge above this to protect battery

# --- Deadline ---
MUST_READY_BY_HOUR   = 7    # Car must be charged by this hour (24h Danish local time)
                             # e.g. 7 means it must be ready by 07:00
# ============================================================

# ── Derived values ────────────────────────────────────────
energy_needed_kwh = BATTERY_CAPACITY_KWH * (TARGET_SOC_PCT - CURRENT_SOC_PCT) / 100
hours_needed      = energy_needed_kwh / CHARGER_POWER_KW

# ── City → price area ─────────────────────────────────────
CITY_AREA_MAP = {
    "aarhus": "DK1", "arhus": "DK1",
    "aalborg": "DK1", "alborg": "DK1",
    "odense": "DK1", "esbjerg": "DK1",
    "horsens": "DK1", "randers": "DK1",
    "kolding": "DK1", "vejle": "DK1",
    "silkeborg": "DK1", "herning": "DK1",
    "viborg": "DK1", "sonderborg": "DK1",
    "holstebro": "DK1", "fredericia": "DK1",
    "hjoerring": "DK1", "frederikshavn": "DK1",
    "skive": "DK1", "thisted": "DK1",
    "ikast": "DK1", "ribe": "DK1",
    "billund": "DK1", "svendborg": "DK1",
    "middelfart": "DK1", "ringkobing": "DK1",
    "haderslev": "DK1", "aabenraa": "DK1", "tonder": "DK1",
    "copenhagen": "DK2", "kobenhavn": "DK2", "cph": "DK2",
    "frederiksberg": "DK2", "roskilde": "DK2",
    "helsingor": "DK2", "hillerod": "DK2",
    "naestved": "DK2", "koege": "DK2",
    "slagelse": "DK2", "holbaek": "DK2",
    "bornholm": "DK2", "roenne": "DK2",
    "ringsted": "DK2", "greve": "DK2",
    "ballerup": "DK2", "gladsaxe": "DK2",
    "lyngby": "DK2", "taastrup": "DK2",
    "allerod": "DK2", "kalundborg": "DK2",
    "soroe": "DK2", "vordingborg": "DK2",
    "haslev": "DK2", "faxe": "DK2",
}

def normalise(s):
    return s.lower().encode("ascii", "ignore").decode().strip()

city_key = normalise(YOUR_CITY)
PRICE_AREA = CITY_AREA_MAP.get(city_key, "DK1")
if city_key not in CITY_AREA_MAP:
    print(f"WARNING: '{YOUR_CITY}' not found. Defaulting to DK1.")
    print("Set PRICE_AREA = 'DK2' below if you live in Copenhagen / Zealand / Bornholm.\n")

area_desc = (
    "West Denmark (Jutland + Funen)"
    if PRICE_AREA == "DK1"
    else "East Denmark (Copenhagen / Zealand / Bornholm)"
)

print(f"Location      : {YOUR_CITY} → {PRICE_AREA} ({area_desc})")
print(f"Battery       : {BATTERY_CAPACITY_KWH} kWh  |  Charger: {CHARGER_POWER_KW} kW")
print(f"Charge needed : {CURRENT_SOC_PCT}% → {TARGET_SOC_PCT}%  ({energy_needed_kwh:.1f} kWh)")
print(f"Charging time : {hours_needed:.1f} hours  ({hours_needed*60:.0f} minutes)")
print(f"Must be ready : {MUST_READY_BY_HOUR:02d}:00 local time")

## Fetch Live CO₂ Data (last 24 hours + forecast)
- `CO2Emis` — actual measured emissions (5-min resolution, ~10-min delay)
- `CO2EmisProg` — Energinet's CO₂ forecast for the next 24–36 hours

In [ ]:
def fetch_energidata(dataset: str, start_dt: datetime, end_dt: datetime,
                     price_area: str, sort_col: str = "Minutes5UTC") -> pd.DataFrame:
    """Generic fetcher for Energi Data Service datasets."""
    url = f"https://api.energidataservice.dk/dataset/{dataset}"
    all_records, offset, batch = [], 0, 10_000

    while True:
        params = {
            "start":  start_dt.strftime("%Y-%m-%dT%H:%M"),
            "end":    end_dt.strftime("%Y-%m-%dT%H:%M"),
            "filter": f'{{"PriceArea":["{price_area}"]}',
            "sort":   f"{sort_col} asc",
            "limit":  batch,
            "offset": offset,
            "timezone": "dk",
        }
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        records = resp.json().get("records", [])
        if not records:
            break
        all_records.extend(records)
        if len(records) < batch:
            break
        offset += batch

    return pd.DataFrame(all_records)


now_utc     = datetime.utcnow()
past_24h    = now_utc - timedelta(hours=24)
future_36h  = now_utc + timedelta(hours=36)

print("Fetching live CO2 data (last 24 h) ...")
df_actual = fetch_energidata("CO2Emis", past_24h, now_utc, PRICE_AREA)
print(f"  CO2Emis records     : {len(df_actual):,}")

print("Fetching CO2 forecast (next 36 h) ...")
try:
    df_forecast = fetch_energidata("CO2EmisProg", now_utc, future_36h, PRICE_AREA)
    print(f"  CO2EmisProg records : {len(df_forecast):,}")
    has_forecast = len(df_forecast) > 0
except Exception as exc:
    print(f"  Forecast unavailable: {exc}")
    df_forecast = pd.DataFrame()
    has_forecast = False

In [ ]:
# ── Process actual data ───────────────────────────────────────────────────────
def process_co2_df(raw: pd.DataFrame, source_label: str) -> pd.DataFrame:
    """Standardise a raw CO2 dataframe."""
    df = raw.copy()
    time_col = "Minutes5DK" if "Minutes5DK" in df.columns else "Minutes5UTC"
    df["ts"] = pd.to_datetime(df[time_col])
    df["hour"] = df["ts"].dt.hour
    df["source"] = source_label

    # Some datasets use 'CO2Emission', others 'CO2Emis'
    co2_col = next(
        (c for c in df.columns if "CO2" in c and "pric" not in c.lower()),
        None
    )
    if co2_col and co2_col != "CO2Emission":
        df["CO2Emission"] = df[co2_col]

    return df[["ts", "hour", "CO2Emission", "source"]].dropna(subset=["CO2Emission"])


df_a = process_co2_df(df_actual,  "actual")

if has_forecast:
    df_f = process_co2_df(df_forecast, "forecast")
    df_combined = pd.concat([df_a, df_f], ignore_index=True).sort_values("ts")
else:
    df_combined = df_a.copy()
    print("Note: No forecast data available — showing actual data only.")

# ── Global CO2 range for colour scale ──────────────────────────────────────
co2_min = df_combined["CO2Emission"].quantile(0.02)
co2_max = df_combined["CO2Emission"].quantile(0.98)

def co2_to_green(val):
    """Map a CO2 value to a 0-100 green score (higher = greener)."""
    return float(np.clip(100 * (1 - (val - co2_min) / (co2_max - co2_min + 1e-9)), 0, 100))

def co2_to_colour(val):
    """Map CO2 value to a matplotlib colour (green = low, red = high)."""
    norm = np.clip((val - co2_min) / (co2_max - co2_min + 1e-9), 0, 1)
    return plt.cm.RdYlGn_r(norm)

df_combined["green_score"] = df_combined["CO2Emission"].apply(co2_to_green)

print(f"Combined dataset: {len(df_combined):,} records")
print(f"Time range: {df_combined['ts'].min()} → {df_combined['ts'].max()}")
df_combined.tail(3)

## Current Status

In [ ]:
latest = df_a.sort_values("ts").iloc[-1]
latest_co2   = latest["CO2Emission"]
latest_score = co2_to_green(latest_co2)
latest_ts    = latest["ts"]

if latest_score >= 70:
    status_emoji = "EXCELLENT"
    advice       = "Grid is very green right now. Start charging!"
elif latest_score >= 50:
    status_emoji = "GOOD"
    advice       = "Decent renewable conditions. Good time to charge."
elif latest_score >= 30:
    status_emoji = "FAIR"
    advice       = "Moderate CO2. Wait for a greener window if you can."
else:
    status_emoji = "POOR"
    advice       = "High CO2 right now. Avoid charging if possible."

print("=" * 50)
print(f"  CURRENT GRID STATUS — {PRICE_AREA}")
print("=" * 50)
print(f"  Time        : {latest_ts.strftime('%Y-%m-%d %H:%M')} (Danish local)")
print(f"  CO2         : {latest_co2:.0f} g/kWh")
print(f"  Green score : {latest_score:.0f} / 100  [{status_emoji}]")
print(f"  Advice      : {advice}")
print("=" * 50)

## Last 24 Hours + Forecast Timeline
The colour of each bar shows how green the electricity was/is.  
**Dashed line** = transition from measured data to forecast.

In [ ]:
# Resample to hourly means for a cleaner plot
df_hourly = (
    df_combined.set_index("ts")
    ["CO2Emission"]
    .resample("1h")
    .mean()
    .reset_index()
    .dropna()
)
df_hourly["green_score"] = df_hourly["CO2Emission"].apply(co2_to_green)
df_hourly["colour"]      = df_hourly["CO2Emission"].apply(co2_to_colour)

fig, ax = plt.subplots(figsize=(16, 5))

for i, row in df_hourly.iterrows():
    ax.bar(row["ts"], row["CO2Emission"],
           width=pd.Timedelta(minutes=55),
           color=row["colour"],
           edgecolor="white", linewidth=0.3,
           align="edge")

# Mark forecast boundary
if has_forecast:
    ax.axvline(df_a["ts"].max(), color="navy", linestyle="--", linewidth=1.5,
               label="Now (forecast starts)")
    ax.text(df_a["ts"].max(), ax.get_ylim()[1] * 0.95,
            " Forecast →", color="navy", fontsize=9, va="top")

# Mark "now"
ax.axvline(latest_ts, color="black", linestyle=":", linewidth=1.5, label="Now")

ax.set_ylabel("CO₂ (g/kWh)")
ax.set_title(f"{PRICE_AREA} — CO₂ Intensity: Last 24 h + Forecast  (green = clean, red = dirty)")

# Colour bar legend
norm = plt.Normalize(co2_min, co2_max)
sm   = plt.cm.ScalarMappable(cmap="RdYlGn_r", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="g CO₂/kWh", shrink=0.7, pad=0.01)
ax.legend()
plt.tight_layout()
plt.show()

## Optimal Charging Windows
Given how much you need to charge and when it must be ready,
find the **consecutive window with the lowest average CO₂** in the available data.

In [ ]:
# ── Build hourly slots from now to the deadline ────────────────────────────
now_local = df_a["ts"].max()           # latest data point ≈ now
hours_until_deadline = (MUST_READY_BY_HOUR - now_local.hour) % 24
if hours_until_deadline == 0:
    hours_until_deadline = 24          # same hour = next day

deadline_ts = now_local.replace(
    minute=0, second=0, microsecond=0
) + timedelta(hours=hours_until_deadline)

# Hourly averages over the planning window
future_data = df_combined[
    (df_combined["ts"] >= now_local) &
    (df_combined["ts"] <= deadline_ts)
].copy()

if future_data.empty:
    # Fall back to what we have
    future_data = df_combined[df_combined["ts"] >= now_local].copy()

hourly_plan = (
    future_data.set_index("ts")["CO2Emission"]
    .resample("1h")
    .mean()
    .dropna()
    .reset_index()
)
hourly_plan["green_score"] = hourly_plan["CO2Emission"].apply(co2_to_green)

print(f"Planning window : {now_local.strftime('%H:%M')} → {deadline_ts.strftime('%d %b %H:%M')}")
print(f"Charging time   : {hours_needed:.1f} h  ({int(np.ceil(hours_needed))} full hours needed)")
print(f"Hourly slots    : {len(hourly_plan)}")
hourly_plan.head()

In [ ]:
# ── Sliding window search for optimal start ───────────────────────────────
n_slots   = int(np.ceil(hours_needed))
n_hours   = len(hourly_plan)

if n_hours < n_slots:
    print(f"WARNING: Only {n_hours} hour(s) of data available but need {n_slots}. ")
    print("Using all available data.")
    n_slots = n_hours

best_start_idx  = 0
best_mean_co2   = float("inf")

for i in range(n_hours - n_slots + 1):
    window_co2 = hourly_plan["CO2Emission"].iloc[i : i + n_slots].mean()
    if window_co2 < best_mean_co2:
        best_mean_co2  = window_co2
        best_start_idx = i

best_window    = hourly_plan.iloc[best_start_idx : best_start_idx + n_slots]
best_start_ts  = best_window["ts"].iloc[0]
best_end_ts    = best_window["ts"].iloc[-1] + timedelta(hours=1)
best_co2_avg   = best_window["CO2Emission"].mean()
best_score_avg = best_window["green_score"].mean()

# Worst window (for comparison)
worst_start_idx = 0
worst_mean_co2  = float("-inf")
for i in range(n_hours - n_slots + 1):
    window_co2 = hourly_plan["CO2Emission"].iloc[i : i + n_slots].mean()
    if window_co2 > worst_mean_co2:
        worst_mean_co2  = window_co2
        worst_start_idx = i
worst_window = hourly_plan.iloc[worst_start_idx : worst_start_idx + n_slots]

print("\n" + "=" * 58)
print(f"  OPTIMAL CHARGING RECOMMENDATION — {PRICE_AREA}")
print("=" * 58)
print(f"  Start charging : {best_start_ts.strftime('%A %d %b  %H:%M')}")
print(f"  Stop charging  : {best_end_ts.strftime('%A %d %b  %H:%M')}")
print(f"  Duration       : {hours_needed:.1f} h  ({energy_needed_kwh:.1f} kWh)")
print(f"  Avg CO2        : {best_co2_avg:.0f} g/kWh  (green score: {best_score_avg:.0f}/100)")
print("=" * 58)
print(f"  (Worst window avg : {worst_mean_co2:.0f} g/kWh — avoided {worst_mean_co2-best_co2_avg:.0f} g/kWh extra)")
print(f"  CO2 saving vs worst window: {(worst_mean_co2-best_co2_avg)*energy_needed_kwh/1000:.2f} kg CO2")

## Charging Schedule Visualisation
Each bar is one hour. **Green = optimal charging window**, grey = other hours.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={"hspace": 0.45})

# ── Top: CO2 intensity with optimal window highlighted ────────────────────
for _, row in hourly_plan.iterrows():
    in_window = best_start_ts <= row["ts"] < best_end_ts
    colour = co2_to_colour(row["CO2Emission"]) if in_window else (0.78, 0.78, 0.78, 0.7)
    ax1.bar(row["ts"], row["CO2Emission"],
            width=pd.Timedelta(minutes=55),
            color=colour,
            edgecolor="white", linewidth=0.4,
            align="edge")

ax1.axvline(latest_ts, color="black", linestyle=":", linewidth=1.5, label="Now")
ax1.axvline(deadline_ts, color="red", linestyle="--", linewidth=1.5,
            label=f"Deadline {MUST_READY_BY_HOUR:02d}:00")

# Highlight optimal window
ax1.axvspan(best_start_ts, best_end_ts, alpha=0.15, color="green", zorder=0,
            label=f"Optimal window ({best_start_ts.strftime('%H:%M')}–{best_end_ts.strftime('%H:%M')})")

ax1.set_ylabel("CO₂ (g/kWh)")
ax1.set_title(f"{PRICE_AREA} — Planning Window: CO₂ Intensity  (green window = recommended charging time)")
ax1.legend(loc="upper right", fontsize=9)

norm = plt.Normalize(co2_min, co2_max)
sm   = plt.cm.ScalarMappable(cmap="RdYlGn_r", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax1, label="g CO₂/kWh", shrink=0.7, pad=0.01)

# ── Bottom: Green score bar chart ────────────────────────────────────────
for _, row in hourly_plan.iterrows():
    in_window = best_start_ts <= row["ts"] < best_end_ts
    colour = co2_to_colour(row["CO2Emission"]) if in_window else (0.78, 0.78, 0.78, 0.7)
    ax2.bar(row["ts"], row["green_score"],
            width=pd.Timedelta(minutes=55),
            color=colour,
            edgecolor="white", linewidth=0.4,
            align="edge")

ax2.axvline(latest_ts, color="black", linestyle=":", linewidth=1.5)
ax2.axvline(deadline_ts, color="red", linestyle="--", linewidth=1.5)
ax2.axhline(50, color="orange", linestyle="-.", linewidth=1, label="Score 50 threshold")
ax2.axvspan(best_start_ts, best_end_ts, alpha=0.15, color="green", zorder=0)
ax2.set_ylabel("Green score (0=dirty, 100=cleanest)")
ax2.set_ylim(0, 105)
ax2.set_title("Green Score per Hour")
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

## Hour-by-Hour Breakdown
Full table of the planning window — sorted by green score.

In [ ]:
hourly_plan["time_label"] = hourly_plan["ts"].dt.strftime("%a %d %b  %H:00")
hourly_plan["in_window"]  = hourly_plan["ts"].apply(
    lambda t: "CHARGE" if best_start_ts <= t < best_end_ts else "-"
)

display_cols = ["time_label", "CO2Emission", "green_score", "in_window"]
display_df   = hourly_plan[display_cols].rename(columns={
    "time_label":   "Hour",
    "CO2Emission":  "CO2 (g/kWh)",
    "green_score":  "Green Score",
    "in_window":    "Action",
})
display_df["CO2 (g/kWh)"] = display_df["CO2 (g/kWh)"].round(0).astype(int)
display_df["Green Score"] = display_df["Green Score"].round(1)

print(display_df.to_string(index=False))

## Tips & Notes

- **Smart charging**: Most modern EVs (Tesla, VW ID, Hyundai Ioniq, etc.) let you set a **departure time** — the car will automatically charge during the cheapest/greenest window overnight. Set that departure time to `MUST_READY_BY_HOUR`.
- **DK1 vs DK2**: DK1 (west) has more wind (especially offshore), so it tends to be greener during high-wind periods. DK2 (east) is more connected to the Swedish nuclear/hydro grid.
- **Night charging**: In Denmark, wind often peaks at night when consumption is low, making late-night/early-morning typically the greenest charging window.
- **Summer vs winter**: Summer adds solar (peak ~11:00–15:00), which can make midday surprisingly green in DK1.
- **Re-run this notebook daily**: The optimal window changes based on current wind/solar forecasts. Integrate with a scheduler (cron / Task Scheduler) for automatic daily recommendations.
- **CO₂EmisProg dataset**: If the forecast cells returned no data, the API may be temporarily unavailable. The actual `CO2Emis` data is always current. Re-run later or use notebook `01_historical_renewable_patterns.ipynb` for a reliable default schedule based on historical patterns.